**12장 – 텐서플로를 사용한 사용자 정의 모델과 훈련**

### 설정

먼저 몇 개의 모듈을 임포트합니다. 맷플롯립 그래프를 인라인으로 출력하도록 만들고 그림을 저장하는 함수를 준비합니다. 또한 파이썬 버전이 3.5 이상인지 확인합니다(파이썬 2.x에서도 동작하지만 곧 지원이 중단되므로 파이썬 3을 사용하는 것이 좋습니다). 사이킷런 버전이 0.20 이상인지와 텐서플로 버전이 2.0 이상인지 확인합니다.

In [ ]:
# 파이썬 ≥3.5 필수
import sys
assert sys.version_info >= (3, 5)

# 코랩의 경우 나눔 폰트를 설치합니다.
if 'google.colab' in sys.modules:
    !sudo apt-get -qq -y install fonts-nanum
    import matplotlib.font_manager as fm
    font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
    for fpath in font_files:
        fm.fontManager.addfont(fpath)

# 사이킷런 ≥0.20 필수
import sklearn
assert sklearn.__version__ >= "0.20"

try:
    # %tensorflow_version은 코랩 명령입니다.
    %tensorflow_version 2.x
except Exception:
    pass

# 이 노트북은 텐서플로 ≥2.4이 필요합니다
# 2.x 버전은 대부분 동일한 결과를 만들지만 몇 가지 버그가 있습니다.
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.11"

# 공통 모듈 임포트
import numpy as np
import os

# 노트북 실행 결과를 동일하게 유지하기 위해
np.random.seed(42)
tf.random.set_seed(42)

# 깔끔한 그래프 출력을 위해
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# 그림을 저장할 위치
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "deep"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("그림 저장:", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

## 사용자 정의 층

Sample data California Housing : https://www.kaggle.com/datasets/camnugent/california-housing-prices


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing(as_frame=True)
housing

In [ ]:
housing.frame

In [ ]:
import matplotlib.pyplot as plt

housing.frame.hist(figsize=(12, 10), bins=30, color='blue', edgecolor="orange")
plt.subplots_adjust(hspace=0.7, wspace=0.4)

In [ ]:
import seaborn as sns
sns.set(rc={'figure.figsize':(12,8)})

sns.scatterplot(data=housing.frame, x="Longitude", y="Latitude",
                size="MedHouseVal", hue="MedHouseVal",
                palette="viridis")
plt.legend(title="MedHouseVal", bbox_to_anchor=(1.05, 0.95),
           loc="upper left")
_ = plt.title("Median house value depending of\n their spatial location")

참고 : https://www.google.com/maps/place/California,+USA/@37.2293576,-111.4151446,6z/data=!4m6!3m5!1s0x808fb9fe5f285e3d:0x8b5109a227086f55!8m2!3d36.778261!4d-119.4179324!16zL20vMDFuN3E?entry=ttu&g_ep=EgoyMDI0MTAwNy4xIKXMDSoASAFQAw%3D%3D

In [ ]:
import seaborn as sns
sns.set(rc={'figure.figsize':(12,8)})

sns.scatterplot(data=housing.frame, x="Longitude", y="Latitude",
                size="MedInc", hue="MedInc",
                palette="viridis")
plt.legend(title="MedInc", bbox_to_anchor=(1.05, 0.95),
           loc="upper left")
_ = plt.title("Median Income value depending of\n their spatial location")

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data.to_numpy(), housing.target.to_numpy().reshape(-1, 1), random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

StandardScaler

z = (x - $\mu$) / $\sigma$

where $\mu$ is the mean of the training samples or zero if with_mean=False, and $\sigma$ is the standard deviation of the training samples

### **사용자 정의 Layer 1**

가중치가 없는 Input Layer나  Activation layer는 keras.layers.Lambda를 상속해서 만들고, 가중치가 있는 레이어는 keras.layers.Layer를 상속 받아, 사용자 정의 레이어를 만들수 있다.

In [ ]:
exponential_layer = keras.layers.Lambda(lambda x: tf.exp(x))

In [ ]:
exponential_layer(tf.constant([-1., 0., 1., 2., 3.]))

회귀 모델이 예측할 값이 양수이고 스케일이 매우 다른 경우 (예를 들어, 0.001, 10., 10000) 출력층에 지수 함수($e^x$)를 추가하면 유용할 수 있습니다:

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
input_shape = X_train.shape[1:]
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="relu", input_shape=input_shape),
    keras.layers.Dense(1),
    exponential_layer
])
model.compile(loss="mse", optimizer="sgd")
model.fit(X_train_scaled, y_train, epochs=5,
          validation_data=(X_valid_scaled, y_valid))
model.evaluate(X_test_scaled, y_test)

### **사용자 정의 Layer 2 - Dense Layer**

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

Dense층의 개념을 위해 구현을 해보자.

Weights 즉 가중치를 가지는 Layer이기때문에 keras.layers.Layer로 부터 상속받아야 한다.

In [ ]:
class MyDense(tf.keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, batch_input_shape):
        self.kernel = self.add_weight(
            name="kernel", shape=[batch_input_shape[-1], self.units],
            initializer="he_normal")
        self.bias = self.add_weight(
            name="bias", shape=[self.units], initializer="zeros")

    def call(self, X):
        return self.activation(X @ self.kernel + self.bias)

    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "units": self.units,
                "activation": tf.keras.activations.serialize(self.activation)}

In [ ]:
model = keras.models.Sequential([
    keras.layers.Input(shape=input_shape),
    MyDense(30, activation="relu"),
    MyDense(1)
])

In [ ]:
model.compile(loss="mse", optimizer="nadam")
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))
model.evaluate(X_test_scaled, y_test)

In [ ]:
keras.utils.plot_model(model, "MyDense.png", show_shapes=True)

훈련된 모델을 저장하고 로딩 해봅시다.

In [ ]:
model.save("my_model_with_a_custom_layer.keras")

In [ ]:
model = keras.models.load_model("my_model_with_a_custom_layer.keras",
                                custom_objects={"MyDense": MyDense})

이전 시간에 Huber Loss 함수를 사용한 모델에서도 Loading 할때, 함수명과 실제 함수를 표시 하여 매핑을 하였는데, 저장된 내요을 보면, 이 사용자 정의 함수명 혹은 사용자 정의 클래스 명이 들어 있는걸 확인 할 수 있습니다.

In [ ]:
!cat my_model_with_a_custom_layer.keras

### **사용자 정의 Layer 3**

In [ ]:
class MyMultiLayer(tf.keras.layers.Layer):
    def call(self, X):
        X1, X2 = X
        print("X1.shape: ", X1.shape ," X2.shape: ", X2.shape)  # extra code
        return X1 + X2, X1 * X2, X1 / X2

사용자 정의 층은 다음처럼 함수형 API를 사용해 호출할 수 있습니다:

In [ ]:
inputs1 = keras.layers.Input(shape=[2])
inputs2 = keras.layers.Input(shape=[2])
MyMultiLayer()((inputs1, inputs2))

`call()` 메서드는 심볼릭 입력을 받습니다. 이 입력의 크기는 부분적으로만 지정되어 있습니다(이 시점에서는 배치 크기를 모릅니다. 그래서 첫 번째 차원이 None입니다):

사용자 층에 실제 데이터를 전달할 수도 있습니다. 이를 테스트하기 위해 각 데이터셋의 입력을 각각 네 개의 특성을 가진 두 부분으로 나누겠습니다:

In [ ]:
def split_data(data):
    columns_count = data.shape[-1]
    half = columns_count // 2
    return data[:, :half], data[:, half:]

X_train_scaled_A, X_train_scaled_B = split_data(X_train_scaled)
X_valid_scaled_A, X_valid_scaled_B = split_data(X_valid_scaled)
X_test_scaled_A, X_test_scaled_B = split_data(X_test_scaled)

# 분할된 데이터 크기 출력
X_train_scaled.shape, X_train_scaled_A.shape, X_train_scaled_B.shape

크기가 완전하게 지정된 것을 볼 수 있습니다:

In [ ]:
MyMultiLayer()((X_train_scaled_A, X_train_scaled_B))

In [ ]:
X1, X2 = np.array([[3., 6.], [2., 7.]]), np.array([[6., 12.], [4., 3.]])
MyMultiLayer()((X1, X2))

함수형 API를 사용해 완전한 모델을 만들어 보겠습니다(이 모델은 간단한 예제이므로 놀라운 성능을 기대하지 마세요):

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

input_A = keras.layers.Input(shape=[X_train_scaled_A.shape[-1]])
input_B = keras.layers.Input(shape=[X_train_scaled_B.shape[-1]])
hidden = MyMultiLayer()((input_A, input_B))
hidden_A = keras.layers.Dense(30, activation='selu')(hidden[0])
hidden_B = keras.layers.Dense(30, activation='selu')(hidden[1])
concat = keras.layers.Concatenate()((hidden_A, hidden_B))
output = keras.layers.Dense(1)(concat)
model = keras.models.Model(inputs=[input_A, input_B], outputs=[output])

In [ ]:
model.compile(loss='mse', optimizer='nadam')

In [ ]:
model.fit((X_train_scaled_A, X_train_scaled_B), y_train, epochs=2,
          validation_data=((X_valid_scaled_A, X_valid_scaled_B), y_valid))

In [ ]:
keras.utils.plot_model(model, "MyMultiLayer.png", show_shapes=True)

### **사용자 정의 Layer 4**

훈련과 테스트에서 다르게 동작하는 층을 만들어 보죠:

가령 훈련중에는 가우시안 노이즈를 입력값에 추가하고, 예측시에는 노이즈를 추가 하지 않는 레이어를 만들어 봅시다.


In [ ]:
class AddGaussianNoise(keras.layers.Layer):
    def __init__(self, stddev, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev

    def call(self, X, training=None):
        if training:
            noise = tf.random.normal(tf.shape(X), stddev=self.stddev)
            return X + noise
        else:
            return X

    def compute_output_shape(self, batch_input_shape):
        return batch_input_shape

다음은 사용자 정의 층을 사용하는 간단한 모델입니다:

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    AddGaussianNoise(stddev=1.0),
    keras.layers.Dense(30, activation="selu"),
    keras.layers.Dense(1)
])

In [ ]:
model.compile(loss="mse", optimizer="nadam")
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))
model.evaluate(X_test_scaled, y_test)

## 사용자 정의 **모델**

지금까지 사용자 정의 층을 만들어 보았는데, 이제 사용자 정의 모델을 만들어 봅시다.

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
X_new_scaled = X_test_scaled

In [ ]:
class ResidualBlock(keras.layers.Layer):
    def __init__(self, n_layers, n_neurons, **kwargs):
        super().__init__(**kwargs)
        self.hidden = [keras.layers.Dense(n_neurons, activation="elu",
                                          kernel_initializer="he_normal")
                       for _ in range(n_layers)]

    def call(self, inputs):
        Z = inputs
        for layer in self.hidden:
            Z = layer(Z)
        return inputs + Z

In [ ]:
class ResidualRegressor(tf.keras.Model):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.output_dim = output_dim
        self.hidden1 = tf.keras.layers.Dense(30, activation="relu",
                                             kernel_initializer="he_normal")
        self.block1 = ResidualBlock(2, 30)
        self.block2 = ResidualBlock(2, 30)
        self.out = tf.keras.layers.Dense(output_dim)

    def call(self, inputs):
        Z = self.hidden1(inputs)
        for _ in range(1 + 3):
            Z = self.block1(Z)
        Z = self.block2(Z)
        return self.out(Z)

    def get_config(self):
        config = super().get_config()
        config.update({"output_dim": self.output_dim})  # Include output_dim in config
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)  # Create instance using config

In [ ]:
model = ResidualRegressor(1)
model.compile(loss="mse", optimizer="nadam")
history = model.fit(X_train_scaled, y_train, epochs=5)
score = model.evaluate(X_test_scaled, y_test)
y_pred = model.predict(X_new_scaled)

In [ ]:
model.save("my_custom_model.keras")

In [ ]:
model = keras.models.load_model("my_custom_model.keras", custom_objects={"ResidualRegressor": ResidualRegressor})

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=5)

앞의 모델을 시퀀셜 API를 사용하여 모델을 정의할 수 있습니다:

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
block1 = ResidualBlock(2, 30)
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="elu", kernel_initializer="he_normal"),
    block1,
    block1,
    block1,
    block1,
    ResidualBlock(2, 30),
    keras.layers.Dense(1)
])

In [ ]:
model.compile(loss="mse", optimizer="nadam")
history = model.fit(X_train_scaled, y_train, epochs=5)
score = model.evaluate(X_test_scaled, y_test)
y_pred = model.predict(X_new_scaled)

In [ ]:
keras.utils.plot_model(model, "ResidualRegressorSequential.png", show_shapes=True)

## 자동 미분을 사용하여 그레이디언트 계산하기

**slope와 gradient**

### GradientTape
y = 3 * $x^{2}$ + 3 일때

dy/dx = 3 * 2 * $x$ 가 된다



x가 7일때 미분에 의해 나온 기울기 식에 대입하면  $3 * 3 * 7 = 42$이다.



In [ ]:
# 수학적 미분은 a * 2 * x
3 * 2 * 7

혹은 x를 조금 변경하여 나온 y값의 크기로 계산 할 수 도 있다.

x가 7 일때 y와, x가 0.001 만큼 증가 할때 y의 값을 구해 기울기를 구할 수도 있다.

In [ ]:
y1 = 3 * 7 ** 2 + 3
y2 = 3 * (7 + 0.001) ** 2 + 3
(y2 - y1) / 0.001

그런데 식을 미분하지 않고, 계산을 기록하여 자동으로 미분하여 주는 기능을 GradientTape를 통해 구현 할수 있다

In [ ]:
x = tf.Variable(7.0)
a = 3
b = 3
with tf.GradientTape() as tape:
  y = a * x ** 2 +  b

dy_dx  = tape.gradient(y, x)

In [ ]:
print(dy_dx)

### 변수가 x 하나가 아닌 여러개 일때
예) z = 3 * $w_1^{2}$ + 2 * $w_1$ * $w_2$ 의 함수가 있다고 하자.



변수가 $w_1$, $w_2$ 두개이므로, $w_1$과 $w_2$에 대해 편미분을 하면

* $\frac{\partial z}{\partial w_1} = 6 \ast  w_1 + 2 \ast w_2$
* $\frac{\partial z}{\partial w_2} =  2 \ast w_1$

가 되고, 예를 들어 $(w_1, w_2)$가 (5, 3) 인경우 그레디언트는 (36, 10)이 된다.

In [ ]:
def f(w1, w2):
    return 3 * w1 ** 2 + 2 * w1 * w2

각 항별로 변화가 일어 날때, 차이를 보기위해 eps 에 작은 값을 지정하여, $w_1$과 $w_2$에 대해 각각 변화 시켜 보자.

$w_1$ 에 대한 기울기

In [ ]:
w1, w2 = 5, 3
eps = 1e-6
(f(w1 + eps, w2) - f(w1, w2)) / eps

$w_2$에 대한 기울기

In [ ]:
(f(w1, w2 + eps) - f(w1, w2)) / eps

이렇게 주어진 위치에 대해 값을 구할수 있지만, 머신러닝에서는 파라미터마다 한번씩 수행을 하면 너무 많은 계산이라 적용이 어렵다.

대신 자동 미분을 사용하여 해결 할수 있다.

In [ ]:
w1, w2 = tf.Variable(5.), tf.Variable(3.)
with tf.GradientTape() as tape:
    z = f(w1, w2)

gradients = tape.gradient(z, [w1, w2])

In [ ]:
gradients

GradientTape를 작성할때, parameter는 tv.Variable이어야한다. 가령 tf.constant를 사용할 경우 아래와 같이 작동이 되지 않는걸 확일 할 수 있다

In [ ]:
c1, c2 = tf.constant(5.), tf.constant(3.)
with tf.GradientTape() as tape:
    z = f(c1, c2)

gradients = tape.gradient(z, [c1, c2])

In [ ]:
gradients

기본적으로 테이프는 변수가 포함된 연산만을 기록한다.

그런데, 주어진 parameter가 constant인경우 아래와 같이 tape.watch를 사용하여 어떤 Tensor라도 감시하여 기록하도록 강제하여 계산 할 수 있다.

In [ ]:
with tf.GradientTape() as tape:
    tape.watch(c1)
    tape.watch(c2)
    z = f(c1, c2)

gradients = tape.gradient(z, [c1, c2])

In [ ]:
gradients

## 사용자 정의 훈련 반복

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
def print_status_bar(iteration, total, loss, metrics=None):
    metrics = " - ".join(["{}: {:.4f}".format(m.name, m.result())
                         for m in [loss] + (metrics or [])])
    end = "" if iteration < total else "\n"
    print("\r{}/{} - ".format(iteration, total) + metrics,
          end=end)

In [ ]:
import time

mean_loss = keras.metrics.Mean(name="loss")
mean_square = keras.metrics.Mean(name="mean_square")
for i in range(1, 50 + 1):
    loss = 1 / i
    mean_loss(loss)
    mean_square(i ** 2)
    print_status_bar(i, 50, mean_loss, [mean_square])
    time.sleep(0.05)

A fancier version with a progress bar:

In [ ]:
def progress_bar(iteration, total, size=30):
    running = iteration < total
    c = ">" if running else "="
    p = (size - 1) * iteration // total
    fmt = "{{:-{}d}}/{{}} [{{}}]".format(len(str(total)))
    params = [iteration, total, "=" * p + c + "." * (size - p - 1)]
    return fmt.format(*params)

In [ ]:
progress_bar(3500, 10000, size=20)

In [ ]:
def print_status_bar(iteration, total, loss, metrics=None, size=30):
    metrics = " - ".join(["{}: {:.4f}".format(m.name, m.result())
                         for m in [loss] + (metrics or [])])
    end = "" if iteration < total else "\n"
    print("\r{} - {}".format(progress_bar(iteration, total), metrics), end=end)

In [ ]:
mean_loss = keras.metrics.Mean(name="loss")
mean_square = keras.metrics.Mean(name="mean_square")
for i in range(1, 50 + 1):
    loss = 1 / i
    mean_loss(loss)
    mean_square(i ** 2)
    print_status_bar(i, 50, mean_loss, [mean_square])
    time.sleep(0.05)

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
def random_batch(X, y, batch_size=32):
    idx = np.random.randint(len(X), size=batch_size)
    return X[idx], y[idx]

In [ ]:
l2_reg = keras.regularizers.l2(0.05)
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="elu", kernel_initializer="he_normal",
                       kernel_regularizer=l2_reg),
    keras.layers.Dense(1, kernel_regularizer=l2_reg)
])

In [ ]:
n_epochs = 5
batch_size = 32
n_steps = len(X_train) // batch_size
optimizer = keras.optimizers.Nadam(learning_rate=0.01)
loss_fn = keras.losses.MeanSquaredError()
mean_loss = keras.metrics.Mean()
metrics = [keras.metrics.MeanAbsoluteError()]

In [ ]:
for epoch in range(1, n_epochs + 1):
    print("Epoch {}/{}".format(epoch, n_epochs))
    for step in range(1, n_steps + 1):
        X_batch, y_batch = random_batch(X_train_scaled, y_train)
        with tf.GradientTape() as tape:
            y_pred = model(X_batch)
            #print(y_pred.shape, y_batch.shape)
            main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
            loss = tf.add_n([main_loss] + model.losses)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        for variable in model.variables:
            if variable.constraint is not None:
                variable.assign(variable.constraint(variable))
        mean_loss(loss)
        for metric in metrics:
            metric(y_batch, y_pred)
        print_status_bar(step * batch_size, len(y_train), mean_loss, metrics)
    print_status_bar(len(y_train), len(y_train), mean_loss, metrics)
    for metric in [mean_loss] + metrics:
        metric.reset_state()


<img src="https://drive.google.com/uc?export=view&id=1bo-9ZBRWT2opqkAlH6S5QOkIDY1x9hKU" width=900/>

In [ ]:
try:
    from tqdm.notebook import trange
    from collections import OrderedDict
    with trange(1, n_epochs + 1, desc="All epochs") as epochs:
        for epoch in epochs:
            with trange(1, n_steps + 1, desc="Epoch {}/{}".format(epoch, n_epochs)) as steps:
                for step in steps:
                    X_batch, y_batch = random_batch(X_train_scaled, y_train)
                    with tf.GradientTape() as tape:
                        y_pred = model(X_batch)
                        main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
                        loss = tf.add_n([main_loss] + model.losses)
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    for variable in model.variables:
                        if variable.constraint is not None:
                            variable.assign(variable.constraint(variable))
                    status = OrderedDict()
                    mean_loss(loss)
                    status["loss"] = mean_loss.result().numpy()
                    for metric in metrics:
                        metric(y_batch, y_pred)
                        status[metric.name] = metric.result().numpy()
                    steps.set_postfix(status)
            for metric in [mean_loss] + metrics:
                metric.reset_state()
except ImportError as ex:
    print("To run this cell, please install tqdm, ipywidgets and restart Jupyter")

## 텐서플로 함수

---
일반 함수를 계산 그래프를 반영한 함수로 변경하여, 빠른 실행이 가능하게 해준다.

성능을 높이고 이식성이 좋은 모델을 만들려면 tf.function을 사용해 그래프로 변환하여 사용한다.


In [ ]:
import tensorflow as tf

x = tf.random.uniform(shape=[100, 100], minval=-1, maxval=1, dtype=tf.dtypes.float32)

def some_costly_computation(x):
    aux = tf.eye(100, dtype=tf.dtypes.float32)
    result = tf.zeros(100, dtype = tf.dtypes.float32)
    for i in range(1,100):
        aux = tf.matmul(x,aux)/i
        result = result + aux
    return result

%timeit some_costly_computation(x)

tf.eye(num_rows,
    num_columns=None,
    batch_shape=None,
    dtype=tf.dtypes.float32,
    name=None
)

identity matrix(단위행렬)을 만들어 반환 한다.

단위 행렬은 matrix의 대각선의 cell에 1이 있고 나머지는 0인 행렬이다.

In [ ]:
quicker_computation = tf.function(some_costly_computation)
%timeit quicker_computation(x)

아래와 같이 decorator로 작성할 수도 있다.

In [ ]:
@tf.function
def quick_computation(x):
    aux = tf.eye(100, dtype=tf.dtypes.float32)
    result = tf.zeros(100, dtype = tf.dtypes.float32)
    for i in range(1,100):
        aux = tf.matmul(x,aux)/i
        result = result + aux
    return result

%timeit quick_computation(x)

example

In [ ]:
def cube(x):
    return x ** 3

In [ ]:
cube(2)

In [ ]:
cube(tf.constant(2.0))

In [ ]:
tf_cube = tf.function(cube)
tf_cube

In [ ]:
tf_cube(2)

In [ ]:
tf_cube(tf.constant(2.0))

### TF 함수와 콘크리트 함수

tf.function 은 polymorphic 즉 다형성을 가진다.

In [ ]:
concrete_function = tf_cube.get_concrete_function(tf.constant(2.0))
concrete_function.graph

In [ ]:
concrete_function(tf.constant(2.0))

In [ ]:
tf_cube(tf.constant(2.0))

In [ ]:
cint = tf_cube.get_concrete_function(2)
cint.graph

In [ ]:
cint(2)

In [ ]:
cf = quick_computation.get_concrete_function(x)
%timeit cf(x)

In [ ]:
%timeit cf(x)

### 함수 정의와 그래프

In [ ]:
concrete_function.graph

In [ ]:
ops = concrete_function.graph.get_operations()
ops

In [ ]:
pow_op = ops[2]
list(pow_op.inputs)

In [ ]:
pow_op.outputs

In [ ]:
concrete_function.graph.get_operation_by_name('x')

In [ ]:
concrete_function.graph.get_tensor_by_name('Identity:0')

In [ ]:
concrete_function.function_def.signature

In [ ]:
cf.graph.get_operations()

# Tips

### tf.where

In [ ]:
tf.where(False, 2, 3)

### variable은 펑션을 가질수 있다

In [ ]:
def gimeAddFunction():
    def f(a, b):
        return a + b
    return f

In [ ]:
myadd = gimeAddFunction()
myadd(3, 5)

In [ ]:
gimeAddFunction()(7, 9)

### arguments and key word arguments

In [ ]:
def hi(*Names):
    for name in Names:
        print("%s %s" % (name[0], name[1:3]), end=' ')
    print("\n")
    print(type(Names), Names)

In [ ]:
hi('이순신', '안중근')

In [ ]:
def hi2(Names):
    for name in Names:
        print("%s %s" % (name[0], name[1:3]), end=' ')
    print("\n")
    print(type(Names), Names)

In [ ]:
hi2('이순신', '안중근')

In [ ]:
def hi3(**Names):
    for key, value in Names.items():
        print("{0} is {1}".format(key, value))

In [ ]:
hi3(MyName="김철수")

In [ ]:
hi3(MyName="김철수", YourName='이영희')

### Python Decorator

In [ ]:
def f1(func):
    def wrapper(*args, **kwargs):
        print("Started")
        func(*args, **kwargs)
        print("End")
    return wrapper

In [ ]:
@f1
def f(a, b=9):
    print(a, b)

In [ ]:
f('oo', b=10)

In [ ]:
@f1
def f2(a, b):
    print(a + b)

In [ ]:
f2(3, 5)

### Carriage Return

In [ ]:
import time
for i in range(10):
    print('\r', i, end='')
    time.sleep(1)